In [3]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/titanic/train.csv
/kaggle/input/competitions/titanic/test.csv
/kaggle/input/competitions/titanic/gender_submission.csv


In [4]:
import pandas as pd
import numpy as np

In [5]:
df=pd.read_csv('/kaggle/input/competitions/titanic/train.csv')

In [6]:
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [12]:
df=df.drop(columns=['PassengerId','Name','Ticket','Cabin'],axis=1)

In [13]:
df.head()

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,0,3,male,22.0,1,0,7.2500,S
1,1,1,female,38.0,1,0,71.2833,C
2,1,3,female,26.0,0,0,7.9250,S
3,1,1,female,35.0,1,0,53.1000,S
4,0,3,male,35.0,0,0,8.0500,S


In [14]:
df.isnull().sum()

Survived      0
Pclass        0
Sex           0
Age         177
SibSp         0
Parch         0
Fare          0
Embarked      2
dtype: int64

## num_cols = 
### SimpleImputer()
### StandardScaler()
## cat_cols = 
### SimpleImputer()
### OneHotEncoder()

In [24]:
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler,OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

In [18]:
x=df.drop('Survived',axis=1)
y=df['Survived']

In [19]:
num_cols=['Age','Fare']
cat_cols=['Sex','Embarked']

In [44]:
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2)

num_pipe=Pipeline([
    ('num_cols',SimpleImputer(strategy='mean')),
    ('num_cols1',StandardScaler())
])

cat_pipe=Pipeline([
    ('cat_cols',SimpleImputer(strategy='most_frequent')),
    ('cat_cols1',OneHotEncoder(drop='first'))
])

preprocess=ColumnTransformer([
    ('num_pipe',num_pipe,num_cols),
    ('cat_pipe',cat_pipe,cat_cols)
],remainder='passthrough')

# Final Pipeline

pipe=Pipeline([
    ('preprocess',preprocess),
    ('model',LogisticRegression())
])

In [47]:
param_grid = {
    'preprocess__num_pipe__num_cols__strategy': ['mean','median'],
    'preprocess__cat_pipe__cat_cols__strategy': ['most_frequent','constant'],
    'model__C': [0.1,1,10,100]
}

grid=GridSearchCV(pipe,param_grid,cv=10)

grid.fit(x_train,y_train)
print('best_params: ',grid.best_params_)

best_params:  {'model__C': 0.1, 'preprocess__cat_pipe__cat_cols__strategy': 'constant', 'preprocess__num_pipe__num_cols__strategy': 'mean'}


In [48]:
y_pred=grid.predict(x_test)

In [52]:
from sklearn.metrics import accuracy_score

print('Accuracy_score :',accuracy_score(y_test,y_pred))

Accuracy_score : 0.7821229050279329


In [53]:
pipe.fit(x_train,y_train)

/usr/local/lib/python3.12/dist-packages/sklearn/compose/_column_transformer.py:1667: FutureWarning: 
The format of the columns of the 'remainder' transformer in ColumnTransformer.transformers_ will change in version 1.7 to match the format of the other transformers.
At the moment the remainder columns are stored as indices (of type int). With the same ColumnTransformer configuration, in the future they will be stored as column names (of type str).
To use the new behavior now and suppress this warning, use ColumnTransformer(force_int_remainder_cols=False).

  warnings.warn(


Pipeline(steps=[('preprocess',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('num_pipe',
                                                  Pipeline(steps=[('num_cols',
                                                                   SimpleImputer()),
                                                                  ('num_cols1',
                                                                   StandardScaler())]),
                                                  ['Age', 'Fare']),
                                                 ('cat_pipe',
                                                  Pipeline(steps=[('cat_cols',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('cat_cols1',
                                                                   OneHotEncoder(drop='first'))]),
                                                  ['Sex', 'Embarked'])])),
                ('model', LogisticRegression())])

In [55]:
pred=pipe.predict(x_test)

print('Accuracy_score :',accuracy_score(y_test,pred))

Accuracy_score : 0.776536312849162
